# SlicePoint quickstart: is this imaging AI a fit for our hospital?

This notebook checks whether an imaging AI model (here, an abdominal aortic aneurysm model) is a good fit for **your** patients and scanners before you turn it on. It compares a sample of your scans with a *reference* (the data the model was checked on) and gives:

- a verdict: **GO**, **BORDERLINE** or **NO-GO**;
- five health checks, each with a plain-language explanation;
- a report you can open in any browser, print to PDF and share with your committee.

**Privacy.** Everything runs on this computer. Nothing is uploaded or sent anywhere, and the report makes no network requests.

**How to use it.** Run the cells from top to bottom (in Jupyter: *Run > Run All Cells*). The only cell you need to edit is Step 2. There is no analysis code in this notebook: every step calls the `slicepoint-core` library, the same code the `slicepoint` command-line tool uses.

## Step 1. Install (once)

Ask whoever set up Python on this computer to run this once, or remove the `#` and run the cell. Until SlicePoint is published, it installs from a copy of the source folder:

```
pip install -e slicepoint/schemas -e slicepoint/core
```

In [ ]:
# %pip install -e ../schemas -e ../core

## Step 2. Tell SlicePoint where your files are

Edit the six lines below.

- `DATA_DIR`: the folder with **your** site's files: `features.parquet`, `predictions.parquet`, and if you have them `labels.parquet` and `embeddings.parquet`.
- `SITE_ID`: a short, stable name for your site, in lowercase (for example `hospital_b`).
- `SITE_NAME`: the name to print on the report (for example `St. Mary's Hospital`). Leave it empty to show the site id made readable (`hospital_b` becomes "Hospital B").
- `REFERENCE`: either a folder with the reference's data files, or a reference profile saved earlier (a folder holding `reference.json`).
- `REFERENCE_SITE_ID`: the reference site's stable id (for example `hospital_a`). It is required when `REFERENCE` is a data folder; SlicePoint never guesses it.
- `OUT_DIR`: where to save the result and the report.

The defaults point at the practice data that comes with SlicePoint (hospital B checked against reference hospital A).

In [ ]:
import os

DATA_DIR = os.environ.get("SLICEPOINT_DATA_DIR", "../../data/practice/hospital_b")
SITE_ID = os.environ.get("SLICEPOINT_SITE_ID", "hospital_b")
SITE_NAME = os.environ.get("SLICEPOINT_SITE_NAME", "")
REFERENCE = os.environ.get("SLICEPOINT_REFERENCE", "../../data/practice/hospital_a")
REFERENCE_SITE_ID = os.environ.get("SLICEPOINT_REFERENCE_SITE_ID", "hospital_a")
OUT_DIR = os.environ.get("SLICEPOINT_OUT_DIR", "slicepoint_output")

## Step 3. Load your data

This reads and checks your files. If a file or column is wrong, the message names it.

In [ ]:
from slicepoint_core import Assessment, ParquetAdapter, ReferenceProfile
from slicepoint_schemas import HACKATHON_DEFAULT_PROFILE

profile = HACKATHON_DEFAULT_PROFILE   # thresholds: decision lines, tolerances, minimum group size
site = ParquetAdapter(SITE_ID).load_tabular(DATA_DIR)
print(f"Loaded {len(site)} scans for {site.site_id}; labels: {site.has_labels}; embeddings: {site.has_embeddings}")

## Step 4. Build or load the reference

The reference describes the population the model was checked on. If `REFERENCE` is a saved profile it is loaded; if it is a data folder, a profile is built from it and saved next to your results so you can reuse it.

In [ ]:
from pathlib import Path

if (Path(REFERENCE) / "reference.json").exists():
    reference = ReferenceProfile.load(REFERENCE)
else:
    assert REFERENCE_SITE_ID, "Set REFERENCE_SITE_ID in Step 2: the reference site's stable id, e.g. hospital_a"
    reference = ReferenceProfile.build(ParquetAdapter(REFERENCE_SITE_ID).load_tabular(REFERENCE), profile)
    reference.save(Path(OUT_DIR) / f"reference_{REFERENCE_SITE_ID}")

## Step 5. Run the assessment

In [ ]:
print(f"Reference {reference.summary.reference_id}: {reference.summary.n_cases} scans")
report = Assessment(site, reference, profile, site_name=SITE_NAME or None).run()
print("Verdict:", report.verdict.final_outcome.value.upper().replace("_", "-"))

## Step 6. Read the verdict and the checks

Each row is one health check. **Status** is `problem`, `watch` or `ok`. A check that could not run shows `skipped` and never counts as passed. **n** is the number of scans the check used; **n_effective** discounts repeat scans of the same person.

In [ ]:
import pandas as pd

summary = report.summary()
pd.DataFrame(summary["checks"]).set_index("id")

The full report is shown below. It is the same file you will save in Step 7.

In [ ]:
from IPython.display import HTML

HTML(report.to_notebook_html(height_px=900))

## Step 7. Save the result and the report

- `result.json` is the complete, versioned result (for your records, or to re-open later). The unvalidated 0 to 100 scores are left out; pass `show_provisional_scores=True` to `write_json` only for internal review.
- `report.html` is the report: double-click to open it in a browser, or print it to PDF.

In [ ]:
result_path = report.write_json(Path(OUT_DIR) / f"{SITE_ID}_result.json")
html_path = report.write_html(Path(OUT_DIR) / f"{SITE_ID}_report.html")
print("Saved:", result_path)
print("Saved:", html_path)

## How to read the report

1. **Verdict first.** NO-GO means at least one check found a problem. BORDERLINE means something needs a closer look or a check could not run. GO means every check passed on this sample.
2. **Check cards.** Each names what failed ("failing sub-factors") and says what it means in plain words.
3. **Performance.** Sensitivity and specificity are shown two ways: counting a missing AI answer as wrong (the main numbers) and leaving those scans out. The ranges are 95% intervals; wide ranges mean the sample is too small to be sure.
4. **Results by group and cases to review** show where the model struggles and which scans to open first.

Results come from a test sample, not a full trial. This report supports a decision and is not medical advice.